# 05 — Post-method train/test evaluation

**Input:** full-cohort TEMPTED and MEFISTO subject scores  
**Does:** creates 20 repeated 70/30 subject-level splits *after* both methods have been fit, then trains the same country classifier on each method's five latent dimensions  
**Output:** split assignments, held-out predictions, and paired performance metrics

The latent models have already seen all microbiome observations. These splits test how well each learned latent representation supports prediction of **FIN vs EST vs RUS** in subjects held out from the classifier.

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

N_SPLITS = 20
TRAIN_SIZE = 0.70
SEED = 2026
COUNTRIES = ["FIN", "EST", "RUS"]

root = Path(".") if Path("data").exists() else Path("..")
tempted_folder = sorted((root / "data" / "tempted").iterdir())[-1]
mefisto_folder = sorted((root / "data" / "mefisto").iterdir())[-1]
output = root / "data" / "evaluation" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

tempted = pd.read_csv(tempted_folder / "subject_scores.csv", dtype={"subject_id": str})
mefisto = pd.read_csv(mefisto_folder / "subject_scores.csv", dtype={"subject_id": str})

subjects = tempted[["subject_id", "country"]].sort_values("subject_id").reset_index(drop=True)
other = mefisto[["subject_id", "country"]].sort_values("subject_id").reset_index(drop=True)

if not subjects.equals(other):
    raise ValueError("TEMPTED and MEFISTO do not contain the same subjects and countries.")

print("TEMPTED:", tempted_folder)
print("MEFISTO:", mefisto_folder)
print("Output:", output)

## Create the splits and evaluate both methods

Each split is generated once and reused for TEMPTED and MEFISTO. Standardization is learned from training subjects only.

In [ ]:
split_rows = []
prediction_rows = []
metric_rows = []
country_rows = []

for split in range(1, N_SPLITS + 1):
    train_subjects, test_subjects = train_test_split(
        subjects,
        train_size=TRAIN_SIZE,
        stratify=subjects["country"],
        random_state=SEED + split,
    )

    train_ids = set(train_subjects["subject_id"])
    test_ids = set(test_subjects["subject_id"])

    if train_ids & test_ids:
        raise ValueError("A subject appears in both train and test.")

    for row in subjects.itertuples(index=False):
        split_rows.append({
            "split": split,
            "subject_id": row.subject_id,
            "country": row.country,
            "set": "train" if row.subject_id in train_ids else "test",
        })

    for method, scores, prefix in [
        ("TEMPTED", tempted, "component_"),
        ("MEFISTO", mefisto, "factor_"),
    ]:
        columns = [f"{prefix}{i}" for i in range(1, 6)]
        train = scores[scores["subject_id"].isin(train_ids)].copy()
        test = scores[scores["subject_id"].isin(test_ids)].copy()

        scaler = StandardScaler()
        x_train = scaler.fit_transform(train[columns])
        x_test = scaler.transform(test[columns])

        classifier = LogisticRegression(max_iter=1000, random_state=SEED)
        classifier.fit(x_train, train["country"])
        predicted = classifier.predict(x_test)
        truth = test["country"].to_numpy()

        metric_rows.append({
            "split": split,
            "method": method,
            "accuracy": accuracy_score(truth, predicted),
            "balanced_accuracy": balanced_accuracy_score(truth, predicted),
            "macro_f1": f1_score(truth, predicted, average="macro"),
        })

        recalls = recall_score(truth, predicted, labels=COUNTRIES, average=None, zero_division=0)
        for country, recall in zip(COUNTRIES, recalls):
            country_rows.append({
                "split": split,
                "method": method,
                "country": country,
                "recall": recall,
            })

        for subject_id, country, guess in zip(test["subject_id"], truth, predicted):
            prediction_rows.append({
                "split": split,
                "method": method,
                "subject_id": subject_id,
                "truth": country,
                "predicted": guess,
            })

In [ ]:
splits = pd.DataFrame(split_rows)
predictions = pd.DataFrame(prediction_rows)
metrics = pd.DataFrame(metric_rows)
country_metrics = pd.DataFrame(country_rows)

for split in range(1, N_SPLITS + 1):
    test_countries = splits[(splits["split"] == split) & (splits["set"] == "test")]["country"]
    if set(test_countries) != set(COUNTRIES):
        raise ValueError(f"Split {split} does not contain all three countries in test.")

splits.to_csv(output / "split_assignments.csv", index=False)
predictions.to_csv(output / "predictions.csv", index=False)
metrics.to_csv(output / "metrics.csv", index=False)
country_metrics.to_csv(output / "per_country_recall.csv", index=False)

summary = metrics.groupby("method")[["accuracy", "balanced_accuracy", "macro_f1"]].agg(["mean", "std"])
summary.to_csv(output / "metric_summary.csv")

print("Saved:", output)
display(summary.round(3))